Importações

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

Carregamento dos dados

In [3]:
DATA_PATH = "./data/"

regular = pd.read_csv(DATA_PATH + "MRegularSeasonCompactResults.csv")
teams = pd.read_csv(DATA_PATH + "MTeams.csv")
seeds = pd.read_csv(DATA_PATH + "MNCAATourneySeeds.csv")

regular.head()
# teams.head()
# seeds.head()


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,20,1228,81,1328,64,N,0
1,1985,25,1106,77,1354,70,H,0
2,1985,25,1112,63,1223,56,H,0
3,1985,25,1165,70,1432,54,H,0
4,1985,25,1192,86,1447,74,H,0


Tratamento dos dados

In [ ]:
# Criar dataframe duplicado para tratar jogos de forma simétrica
win_df = regular[['Season','WTeamID','LTeamID','WScore','LScore']].copy()
win_df['Result'] = 1

lose_df = regular[['Season','LTeamID','WTeamID','LScore','WScore']].copy()
lose_df.columns = ['Season','WTeamID','LTeamID','WScore','LScore']
lose_df['Result'] = 0

data = pd.concat([win_df, lose_df], axis=0)

# Estatísticas por time
team_stats = data.groupby(['Season','WTeamID']).agg({
    'WScore':'mean',
    'LScore':'mean',
    'Result':'mean'
}).reset_index()

team_stats.columns = ['Season','TeamID','AvgScore','AvgAllowed','WinRate']

# Merge stats time A
model_df = data.merge(team_stats, left_on=['Season','WTeamID'], right_on=['Season','TeamID'])
model_df = model_df.rename(columns={
    'AvgScore':'A_AvgScore',
    'AvgAllowed':'A_AvgAllowed',
    'WinRate':'A_WinRate'
})

# Merge stats time B
model_df = model_df.merge(team_stats, left_on=['Season','LTeamID'], right_on=['Season','TeamID'])
model_df = model_df.rename(columns={
    'AvgScore':'B_AvgScore',
    'AvgAllowed':'B_AvgAllowed',
    'WinRate':'B_WinRate'
})

# Diferenças (features principais)
model_df['Diff_Score'] = model_df['A_AvgScore'] - model_df['B_AvgScore']
model_df['Diff_Defense'] = model_df['A_AvgAllowed'] - model_df['B_AvgAllowed']
model_df['Diff_WinRate'] = model_df['A_WinRate'] - model_df['B_WinRate']

# Seleção final
features = ['Diff_Score','Diff_Defense','Diff_WinRate']
X = model_df[features]
y = model_df['Result']

Calculando o baseline

In [ ]:
# Função de cálculo da métrica (Brier Score)
def brier_score(y_true, y_prob):
    return np.mean((y_prob - y_true) ** 2)

# Baseline
baseline_preds = np.full(len(y), 0.5)
print("Baseline Brier Score:", brier_score(y, baseline_preds))

Baseline Brier Score: 0.25


Criação dos Modelos de Predição

In [ ]:
# Modelos
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100)
}

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    scores = []

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)
        probs = model.predict_proba(X_val)[:,1]

        score = brier_score(y_val, probs)
        scores.append(score)

    print(f"{name} - Brier Score médio: {np.mean(scores)}")

LogisticRegression - Brier Score médio: 0.17314460891321626
RandomForest - Brier Score médio: 0.2327279335663454


In [ ]:

# 7. Ensemble (Meta-learning simples)
model1 = LogisticRegression(max_iter=1000)
model2 = RandomForestClassifier(n_estimators=100)

model1.fit(X, y)
model2.fit(X, y)

probs1 = model1.predict_proba(X)[:,1]
probs2 = model2.predict_proba(X)[:,1]

ensemble_probs = (probs1 + probs2) / 2

print("Ensemble Brier Score:", brier_score(y, ensemble_probs))


# 8. Próximos passos
# - Adicionar seeds
# - Adicionar estatísticas detalhadas
# - Testar XGBoost / LightGBM
# - Fazer split temporal (treino até 2024, teste 2025)
